For every significant component, I will answer4  questions without looking at the generated code:

1. What?

What does this component do?

2. Why?

Why does the architecture need it?

3. How?

Conceptually, how does it work?

4. What can go wrong?

What assumptions, failure modes, security problems, data problems or ML problems exist?

Then let AI write the implementation.

That's a much more future-proof workflow.

# 1. What is a smoke test?

A smoke test is a very basic test that checks whether the main parts of a system are working at all.

Think of it like turning on a new car for the first time:

- Does the engine start?
- Does the dashboard work?
- Are the basic systems connected?

You are not testing whether the car can drive 300 km/h yet.

For MedImageForge, the smoke tests check:
```
MedImageForge
     │
     ├── Package exists
     ├── Version exists
     ├── Configuration works
     ├── Paths work
     ├── Dataset exists
     └── Basic foundation is ready
```


```
              MedImageForge
                    │
                    ▼
             Python package
                    │
                    ▼
              __version__
                    │
                    ▼
              Configuration
                    │
                    ▼
              Path resolution
                    │
                    ▼
             Dataset location
                    │
                    ▼
           CT-ICH dataset exists
```

## Test 1 — Does the version exist?
``` python 
def test_version_exists():
    assert __version__
```
This is the simplest test in the file.

What does def test_... mean?

With pytest, functions beginning with:
test_
are automatically discovered as tests.

**"Can Python see my MedImageForge package correctly?"**

In [1]:
import sys

print(sys.executable)

/home/zahra/MedImageForge/.venv/bin/python


In [2]:
from medimageforge import __version__

print("MedImageForge version:", __version__)

MedImageForge version: 0.1.0


## Test 2 — Does the configuration load?
``` python
def test_default_config_loads():
    config = load_config()
    assert config["project"]["name"] == "medimageforge"
    assert "paths" in config and "dataset" in config
```
***f default.yaml is broken, Step 4 will fail.***


In [3]:
from medimageforge.config import load_config

config = load_config()

print(config)

{'project': {'name': 'medimageforge', 'version': '0.1.0'}, 'paths': {'data_dir': 'data', 'raw_dir': 'data/Patients_CT', 'labels_csv': 'data/hemorrhage_diagnosis.csv', 'demographics_csv': 'data/patient_demographics.csv', 'checksums_file': 'data/SHA256SUMS.txt', 'artifacts_dir': 'artifacts'}, 'dataset': {'patient_id_width': 3, 'windows': ['brain', 'bone'], 'mask_suffix': '_HGE_Seg'}, 'logging': {'level': 'INFO'}}


## Test 3 — Do configured paths stay inside the project?
``` python
def test_configured_paths_resolve_inside_project():
    config = load_config()
    for key in config["paths"]:
        p = data_path(config, key)
        assert p.is_absolute()
        assert str(p).startswith(str(PROJECT_ROOT))
```

loop through all configured paths
``` json
paths:
  raw_dir: data/Patients_CT
  labels_csv: data/hemorrhage_diagnosis.csv
  demographics_csv: data/patient_demographics.csv
```
Then the loop will examine:
```
raw_dir
labels_csv
demographics_csv
```

Is the path absolute?

In [4]:
from medimageforge.config import data_path

for key in config["paths"]:
    p = data_path(config, key)

    print(f"{key:20} → {p}")

data_dir             → /home/zahra/MedImageForge/data
raw_dir              → /home/zahra/MedImageForge/data/Patients_CT
labels_csv           → /home/zahra/MedImageForge/data/hemorrhage_diagnosis.csv
demographics_csv     → /home/zahra/MedImageForge/data/patient_demographics.csv
checksums_file       → /home/zahra/MedImageForge/data/SHA256SUMS.txt
artifacts_dir        → /home/zahra/MedImageForge/artifacts


## Test 4 — Is the dataset actually present?
``` python 
def test_dataset_is_present():
    # The CT-ICH dataset must be in place for every later step.
    config = load_config()
    assert data_path(config, "raw_dir").is_dir()
    assert data_path(config, "labels_csv").is_file()
``` 
**Is the CT-ICH dataset actually available?**